<a href="https://colab.research.google.com/github/AksharaBasu/GenAI/blob/main/LLM3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# A1 → ROUTE → A2 LANGGRAPH AI
# A1 = OpenRouter
# A2 = Groq
# ============================================================

!pip install -q -U langgraph langchain-openrouter langchain-groq gradio


# ============================================================
# IMPORTS
# ============================================================

import gradio as gr

from typing import TypedDict

from google.colab import userdata

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END

print("✅ Libraries imported")


# ============================================================
# GET YOUR COLAB SECRETS
# ============================================================

# Your actual secret names are:
# openrouter
# GROQ_API_KEY

OPENROUTER_API_KEY = userdata.get("openrouter")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")


# ============================================================
# CHECK KEYS
# ============================================================

if not OPENROUTER_API_KEY:
    raise ValueError(
        "❌ OpenRouter key not found.\n"
        "Check that the Colab Secret is named exactly: openrouter\n"
        "Also enable Notebook access."
    )

if not GROQ_API_KEY:
    raise ValueError(
        "❌ Groq key not found.\n"
        "Check that the Colab Secret is named exactly: GROQ_API_KEY\n"
        "Also enable Notebook access."
    )

print("✅ OpenRouter API key found")
print("✅ Groq API key found")


# ============================================================
# A1 = OPENROUTER
# ============================================================

llm_a1 = ChatOpenRouter(
    api_key=OPENROUTER_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)

print("✅ A1 = OpenRouter")


# ============================================================
# A2 = GROQ
# ============================================================

llm_a2 = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)

print("✅ A2 = Groq")


# ============================================================
# STATE
# ============================================================

class State(TypedDict):
    question: str
    route: str
    answer_a1: str
    answer_a2: str


print("✅ State created")


# ============================================================
# A1 NODE
# ============================================================

def a1_node(state: State):

    question = state["question"]

    prompt = (
        "You are AI Assistant A1.\n\n"
        "Answer the user's question accurately and clearly.\n"
        "Explain the answer properly.\n"
        "If code is requested, provide working code.\n"
        "If it is an academic question, make it exam-friendly.\n"
        "If steps are required, provide clear steps.\n\n"
        "USER QUESTION:\n"
        + question
    )

    response = llm_a1.invoke(prompt)

    return {
        "answer_a1": str(response.content).strip()
    }


# ============================================================
# ROUTE
# ============================================================

def route_after_a1(state: State):

    return "a2"


# ============================================================
# A2 NODE
# ============================================================

def a2_node(state: State):

    question = state["question"]
    answer_a1 = state["answer_a1"]

    prompt = (
        "You are AI Assistant A2.\n\n"
        "A1 has already generated an answer.\n"
        "You are the final reviewer and answer generator.\n\n"

        "ORIGINAL QUESTION:\n"
        + question
        + "\n\n"

        "A1 ANSWER:\n"
        + answer_a1
        + "\n\n"

        "Your job:\n"
        "1. Check the A1 answer.\n"
        "2. Correct mistakes.\n"
        "3. Add missing important information.\n"
        "4. Remove unnecessary information.\n"
        "5. Make the answer clear.\n"
        "6. Correct code if necessary.\n"
        "7. Make academic answers exam-friendly.\n\n"

        "Give ONLY the final improved answer.\n"
        "Do not mention A1.\n"
        "Do not mention A2.\n"
        "Do not mention LangGraph."
    )

    response = llm_a2.invoke(prompt)

    return {
        "answer_a2": str(response.content).strip()
    }


# ============================================================
# BUILD GRAPH
# ============================================================

builder = StateGraph(State)


# ============================================================
# ADD NODES
# ============================================================

builder.add_node("a1", a1_node)
builder.add_node("a2", a2_node)


# ============================================================
# START → A1
# ============================================================

builder.add_edge(
    START,
    "a1"
)


# ============================================================
# A1 → ROUTE → A2
# ============================================================

builder.add_conditional_edges(
    "a1",
    route_after_a1,
    {
        "a2": "a2"
    }
)


# ============================================================
# A2 → END
# ============================================================

builder.add_edge(
    "a2",
    END
)


# ============================================================
# COMPILE
# ============================================================

graph = builder.compile()

print("✅ Graph compiled successfully")


# ============================================================
# CHAT FUNCTION
# ============================================================

def ask_ai(question):

    if not question or not question.strip():

        return (
            "❌ Please enter a question.",
            "",
            ""
        )

    try:

        initial_state = {
            "question": question.strip(),
            "route": "",
            "answer_a1": "",
            "answer_a2": ""
        }

        result = graph.invoke(initial_state)

        a1_answer = result["answer_a1"]
        a2_answer = result["answer_a2"]

        return (
            "A1 → ROUTE → A2",
            a1_answer,
            a2_answer
        )

    except Exception as e:

        return (
            "❌ ERROR:\n\n" + str(e),
            "",
            ""
        )


# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(
    title="A1 → A2 LangGraph AI"
) as demo:

    gr.Markdown(
        """
# 🤖 A1 → Route → A2 LangGraph AI

### Two LLM Architecture

**OpenRouter → A1 → Route → Groq → A2**

A1 creates the first answer.

The State stores the A1 answer.

The Route sends A1 to A2.

A2 reviews the A1 answer and produces the final answer.
"""
    )

    question_box = gr.Textbox(
        label="📝 Ask your question",
        placeholder="Example: What is LangGraph?",
        lines=4
    )

    ask_button = gr.Button(
        "🚀 Ask AI",
        variant="primary"
    )

    route_output = gr.Textbox(
        label="🔀 Route"
    )

    a1_output = gr.Markdown(
        label="A1 Answer — OpenRouter"
    )

    a2_output = gr.Markdown(
        label="A2 Final Answer — Groq"
    )

    ask_button.click(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            route_output,
            a1_output,
            a2_output
        ]
    )

    question_box.submit(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            route_output,
            a1_output,
            a2_output
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

print("")
print("=" * 60)
print("🚀 LANGGRAPH AI READY")
print("=" * 60)
print("A1     → OpenRouter")
print("Route  → A1 to A2")
print("A2     → Groq")
print("=" * 60)
print("")

demo.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.9 MB/s eta 0:00:00
✅ Libraries imported
✅ OpenRouter API key found
✅ Groq API key found
✅ A1 = OpenRouter
✅ A2 = Groq
✅ State created
✅ Graph compiled successfully

🚀 LANGGRAPH AI READY
A1     → OpenRouter
Route  → A1 to A2
A2     → Groq

Colab notebook detected. This cell will run indefinitely so that